In [1]:
import pandas as pd
import numpy as np

# Load the cleaned dataset
df = pd.read_csv('cleaned_dataset.csv')

# Verify dataset shape and structure
print(f"Loaded dataset shape: {df.shape}")
display(df.head(3))

Loaded dataset shape: (6589, 50)


,CustomerID,Gender,Age,Under30,SeniorCitizen,Married,Dependents,NumberofDependents,Country,State,...,TotalExtraDataCharges,TotalLongDistanceCharges,TotalRevenue,SatisfactionScore,CustomerStatus,ChurnLabel,ChurnScore,CLTV,ChurnCategory,ChurnReason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,...,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,...,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,...,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer


In [2]:
# Check data types of all columns
print("Data types summary:")
print(df.dtypes.value_counts())

# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"\nNumber of Categorical Columns: {len(categorical_cols)}")
print(f"Number of Numerical Columns: {len(numerical_cols)}")

Data types summary:
object     31
int64      11
float64     8
Name: count, dtype: int64

Number of Categorical Columns: 31
Number of Numerical Columns: 19


In [3]:
# ---Encoding Categorical Variables ---
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
le = LabelEncoder()

# Encode categorical columns that have text values
for col in categorical_cols:
    if col != 'CustomerID':  # Keep CustomerID as is for tracking
        df_encoded[col] = df_encoded[col].astype(str)
        df_encoded[col] = le.fit_transform(df_encoded[col])

# ---CLTV Calculation / Feature Engineering ---
# Ensure CLTV is computed or structured properly if not already present, or
# Derive customer lifetime value based on Monthly Charge and Tenure
if 'CLTV' in df_encoded.columns:
    print("CLTV column already exists in the dataset.")
else:
    # Basic business formula for CLTV if needed: (Monthly Charge * Tenure) * adjustment factor
    df_encoded['CLTV'] = df_encoded['MonthlyCharge'] * df_encoded['TenureinMonths']

print(f"Encoded dataset shape: {df_encoded.shape}")
display(df_encoded[['CustomerID', 'MonthlyCharge', 'TenureinMonths', 'CLTV']].head(3))

CLTV column already exists in the dataset.
Encoded dataset shape: (6589, 50)


,CustomerID,MonthlyCharge,TenureinMonths,CLTV
0,8779-QRDMV,39.65,1,5433
1,7495-OOKFY,80.65,8,5302
2,1658-BYGOY,95.45,18,3179


In [4]:
from sklearn.preprocessing import StandardScaler

# ---Handle Target Variable encoding---
if 'CustomerStatus' in df_encoded.columns:
    # Let's map Churned to 1 and Stayed to 0 (assuming those are the main two statuses left)
    df_encoded['Target'] = df_encoded['CustomerStatus'].apply(lambda x: 1 if str(x).lower() in ['churned', '1'] else 0)

# ---Drop unnecessary identifier columns ---
if 'CustomerID' in df_encoded.columns:
    df_encoded = df_encoded.drop(columns=['CustomerID'])

# ---Scale Numerical Features ---
# Select continuous numerical columns for scaling
scale_cols = ['Age', 'TenureinMonths', 'MonthlyCharge', 'TotalRevenue', 'CLTV', 'ChurnScore', 'SatisfactionScore']
existing_scale_cols = [col for col in scale_cols if col in df_encoded.columns]

scaler = StandardScaler()
df_encoded[existing_scale_cols] = scaler.fit_transform(df_encoded[existing_scale_cols])

# Save the final scaled and engineered dataset
df_encoded.to_csv('processed_churn_data.csv', index=False)
print("Advanced Feature Engineering & Scaling completed successfully!")
print(f"Final dataset shape ready for modeling: {df_encoded.shape}")

Advanced Feature Engineering & Scaling completed successfully!
Final dataset shape ready for modeling: (6589, 50)
